# Notebook 3 — Feature Engineering & ML Pipeline Construction
## HMDA 2023 Loan Denial Prediction: From Raw Data → Model-Ready Vectors

**Objective:** Transform 99 raw HMDA columns into a clean, encoded, scaled feature matrix that PySpark ML classifiers can consume directly.

---

## Section 1 — Setup & Data Loading

In [ ]:
# ============================================================
# Imports & SparkSession
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Imputer
)
from pyspark.ml.stat import Summarizer

import pandas as pd
import numpy as np
import json
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("HMDA_2023_FeatureEngineering")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

In [2]:
# ============================================================
# Load Data, Schema & EDA Results
# ============================================================

# ── Paths (adjust for your local environment) ──
PARQUET_PATH   = "../data/processed/hmda_2023.parquet"
SCHEMA_PATH    = "../data/schemas/hmda_schema.json"
EDA_RESULTS    = "../data/processed/eda_results.pkl"
OUTPUT_DIR     = "../data/processed"

# ── Load Parquet ──
df = spark.read.parquet(PARQUET_PATH)
total_rows = df.count()
print(f"Loaded: {total_rows:,} rows × {len(df.columns)} columns")

# ── Load Schema ──
with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)

# Reference lists from schema — these are our ground-truth column classifications
NUMERIC_CONTINUOUS  = schema["expected_dtypes"]["numeric_continuous"]     # 22 cols
CATEGORICAL_NOMINAL = schema["expected_dtypes"]["categorical_nominal"]   # 67 cols
CATEGORICAL_ORDINAL = schema["expected_dtypes"]["categorical_ordinal"]   # 6 cols
LEAKAGE_COLS        = schema["leakage_columns"]["columns"]               # 12 cols

print(f"\nSchema loaded:")
print(f"  Numeric continuous:  {len(NUMERIC_CONTINUOUS)}")
print(f"  Categorical nominal: {len(CATEGORICAL_NOMINAL)}")
print(f"  Categorical ordinal: {len(CATEGORICAL_ORDINAL)}")
print(f"  Leakage columns:     {len(LEAKAGE_COLS)}")

# ── Load EDA Results (optional — graceful fallback) ──
try:
    with open(EDA_RESULTS, "rb") as f:
        eda = pickle.load(f)
    print(f"\nEDA results loaded: {len(eda)} keys")
    print(f"  NZV columns:         {len(eda.get('nzv_cols', []))}")
    print(f"  Multicollinear pairs:{len(eda.get('multicoll_pairs', []))}")
    print(f"  Informative missing: {len(eda.get('informative_missing', []))}")
except FileNotFoundError:
    print("\n⚠ EDA results not found — using schema-based defaults")
    eda = {}

Loaded: 11,483,889 rows × 99 columns

Schema loaded:
  Numeric continuous:  22
  Categorical nominal: 69
  Categorical ordinal: 6
  Leakage columns:     12

EDA results loaded: 12 keys
  NZV columns:         28
  Multicollinear pairs:1
  Informative missing: 60


## Section 1b — Column Name Normalization

In [3]:
# ============================================================
#  Normalize Column Names
# ============================================================

original_cols = df.columns[:]
rename_count = 0

for col_name in original_cols:
    clean = col_name.replace("-", "_")
    if clean != col_name:
        df = df.withColumnRenamed(col_name, clean)
        rename_count += 1

print(f"✓ Normalized {rename_count} column names (hyphen → underscore)")
if rename_count > 0:
    print(f"  Examples of renamed columns:")
    for old in original_cols:
        new = old.replace("-", "_")
        if new != old:
            print(f"    {old:40s} → {new}")
            if rename_count > 8:
                break  # Don't flood output

print(f"\n  Total columns: {len(df.columns)}")

✓ Normalized 0 column names (hyphen → underscore)

  Total columns: 99


## Section 2 — Column Selection: Dropping Useless & Dangerous Columns

In [4]:
# ============================================================
# Define All Columns to Drop
# ============================================================

# A. LEAKAGE
LEAKAGE_DROP = [
    "denial_reason_1", "denial_reason_2", "denial_reason_3", "denial_reason_4",
    "purchaser_type", "rate_spread",
    "total_loan_costs", "total_points_and_fees",
    "origination_charges", "discount_points", "lender_credits",
    "prepayment_penalty_term",
]

# B. IDENTIFIERS — too high cardinality, no predictive generalization
IDENTIFIER_DROP = [
    "lei",            # 5000+ unique lender IDs
    "census_tract",   # 70,000+ unique tract codes
    "county_code",    # 3000+ counties
    "activity_year",  # Single value (2023) — literally zero variance
    "derived_msa_md", # Metro area code — use census tract context cols instead
]

# C. NEAR-ZERO VARIANCE — >95% single value
NZV_DROP = [
    "reverse_mortgage",               # >99% "2" (No)
    "open_end_line_of_credit",        # >95% "2" (No)
    "business_or_commercial_purpose", # >95% "2" (No)
    "negative_amortization",          # >99% "2" (No)
    "interest_only_payment",          # >95% "2" (No)
    "balloon_payment",                # >99% "2" (No)
    "other_nonamortizing_features",   # >95% "2" (No)
    "hoepa_status",                   # >99% "2" — BUT we extract high_cost_flag FIRST
]

# D. REDUNDANT DEMOGRAPHIC DETAIL — use derived_* versions instead
DEMOGRAPHIC_DETAIL_DROP = [
    "applicant_ethnicity_1", "applicant_ethnicity_2", "applicant_ethnicity_3",
    "applicant_ethnicity_4", "applicant_ethnicity_5",
    "co_applicant_ethnicity_1", "co_applicant_ethnicity_2", "co_applicant_ethnicity_3",
    "co_applicant_ethnicity_4", "co_applicant_ethnicity_5",
    "applicant_ethnicity_observed", "co_applicant_ethnicity_observed",
    "applicant_race_1", "applicant_race_2", "applicant_race_3",
    "applicant_race_4", "applicant_race_5",
    "co_applicant_race_1", "co_applicant_race_2", "co_applicant_race_3",
    "co_applicant_race_4", "co_applicant_race_5",
    "applicant_race_observed", "co_applicant_race_observed",
    "applicant_sex", "co_applicant_sex",
    "applicant_sex_observed", "co_applicant_sex_observed",
]

# E. AUS DETAIL — aus_2 through aus_5 are >95% null
AUS_DETAIL_DROP = ["aus_2", "aus_3", "aus_4", "aus_5"]

# F. TARGET SOURCE — we derive `label` from it, then drop
TARGET_SOURCE_DROP = ["action_taken"]

# ── Combine all drops ──
ALL_DROP = list(set(
    LEAKAGE_DROP + IDENTIFIER_DROP + NZV_DROP +
    DEMOGRAPHIC_DETAIL_DROP + AUS_DETAIL_DROP + TARGET_SOURCE_DROP
))

# Only drop columns that actually exist (defensive coding)
existing_drops = [c for c in ALL_DROP if c in df.columns]
missing_drops  = [c for c in ALL_DROP if c not in df.columns]

print(f"Total columns in DataFrame:  {len(df.columns)}")
print(f"Columns to drop:             {len(existing_drops)}")
print(f"  Leakage:              {len([c for c in LEAKAGE_DROP if c in df.columns])}")
print(f"  Identifiers:          {len([c for c in IDENTIFIER_DROP if c in df.columns])}")
print(f"  Near-zero variance:   {len([c for c in NZV_DROP if c in df.columns])}")
print(f"  Demographic detail:   {len([c for c in DEMOGRAPHIC_DETAIL_DROP if c in df.columns])}")
print(f"  AUS detail:           {len([c for c in AUS_DETAIL_DROP if c in df.columns])}")
print(f"  Target source:        {len([c for c in TARGET_SOURCE_DROP if c in df.columns])}")
print(f"Columns remaining:           {len(df.columns) - len(existing_drops)}")

if missing_drops:
    print(f"\n⚠ WARNING: {len(missing_drops)} intended drop columns not found in DataFrame:")
    for c in sorted(missing_drops):
        print(f"    MISSING: {c}")
    print("\n  → Check if these were already dropped in a prior step or renamed.")
    print("  → If this is unexpected, investigate before proceeding!")

# Hard assert on leakage columns — these MUST be dropped
leakage_missing = [c for c in LEAKAGE_DROP if c not in df.columns]
assert len(leakage_missing) == 0, (
    f"CRITICAL: {len(leakage_missing)} LEAKAGE columns not found for dropping! "
    f"This means they either don't exist (schema change) or were renamed: {leakage_missing}. "
    f"Actual columns containing 'denial': {[c for c in df.columns if 'denial' in c]}"
)
print("\n✓ All 12 leakage columns confirmed present and will be dropped.")

Total columns in DataFrame:  99
Columns to drop:             58
  Leakage:              12
  Identifiers:          5
  Near-zero variance:   8
  Demographic detail:   28
  AUS detail:           4
  Target source:        1
Columns remaining:           41

✓ All 12 leakage columns confirmed present and will be dropped.


## Section 3 — Target Variable Creation & Population Filtering

In [5]:
# ============================================================
# Filter to Target Population & Create Label
# ============================================================

start = time.time()

df = df.withColumn(
    "high_cost_flag",
    F.when(F.col("hoepa_status").cast("string") == "1", 1.0).otherwise(0.0)
)

df_filtered = df.filter(F.col("action_taken").isin(1, 3))

df_filtered = df_filtered.withColumn(
    "label",
    F.when(F.col("action_taken") == 3, 1.0).otherwise(0.0)
)
df_eng = df_filtered.drop(*existing_drops)

LEAKAGE_PATTERNS = ["denial_reason", "rate_spread", "purchaser_type",
                    "total_loan_costs", "total_points_and_fees",
                    "origination_charges", "discount_points",
                    "lender_credits", "prepayment_penalty"]
surviving_leakage = [c for c in df_eng.columns
                     for pat in LEAKAGE_PATTERNS if pat in c]
assert len(surviving_leakage) == 0, (
    f"LEAKAGE DETECTED: {len(surviving_leakage)} columns with leakage "
    f"patterns survived the drop: {surviving_leakage}"
)

# ── Verify ──
target_total    = df_eng.count()
denied_count    = df_eng.filter(F.col("label") == 1).count()
originated_count = target_total - denied_count

print(f"✓ Feature engineering DataFrame created in {time.time()-start:.1f}s")
print(f"  Total records:        {target_total:,}")
print(f"  Originated (label=0): {originated_count:,} ({originated_count/target_total*100:.1f}%)")
print(f"  Denied     (label=1): {denied_count:,}   ({denied_count/target_total*100:.1f}%)")
print(f"  Columns remaining:    {len(df_eng.columns)}")
print(f"  Imbalance ratio:      {originated_count/max(denied_count,1):.1f}:1")
print(f"  ✓ Post-drop leakage check PASSED (0 leakage columns remaining)")

# Cache — we'll transform this DataFrame many times
df_eng.cache()
df_eng.count()  # Force materialization
print("✓ DataFrame cached.")

# List remaining columns
print(f"\nRemaining {len(df_eng.columns)} columns:")
for i, c in enumerate(sorted(df_eng.columns), 1):
    print(f"  {i:2d}. {c}")

✓ Feature engineering DataFrame created in 0.9s
  Total records:        7,686,484
  Originated (label=0): 5,691,726 (74.0%)
  Denied     (label=1): 1,994,758   (26.0%)
  Columns remaining:    43
  Imbalance ratio:      2.9:1
  ✓ Post-drop leakage check PASSED (0 leakage columns remaining)


26/03/02 00:35:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✓ DataFrame cached.

Remaining 43 columns:
   1. applicant_age
   2. applicant_age_above_62
   3. applicant_credit_score_type
   4. aus_1
   5. co_applicant_age
   6. co_applicant_age_above_62
   7. co_applicant_credit_score_type
   8. combined_loan_to_value_ratio
   9. conforming_loan_limit
  10. construction_method
  11. debt_to_income_ratio
  12. derived_dwelling_category
  13. derived_ethnicity
  14. derived_loan_product_type
  15. derived_race
  16. derived_sex
  17. ffiec_msa_md_median_family_income
  18. high_cost_flag
  19. income
  20. initially_payable_to_institution
  21. interest_rate
  22. intro_rate_period
  23. label
  24. lien_status
  25. loan_amount
  26. loan_purpose
  27. loan_term
  28. loan_type
  29. manufactured_home_land_property_interest
  30. manufactured_home_secured_property_type
  31. multifamily_affordable_units
  32. occupancy_type
  33. preapproval
  34. property_value
  35. state_code
  36. submission_of_application
  37. total_units
  38. tract_median

## Section 4 — Missing Value Normalization: HMDA's Triple-Missing Problem

In [6]:
# ============================================================
# Normalize Missing Values → null
# ============================================================

MISSING_VALUES = ["Exempt", "NA", "N/A", "Not applicable", "1111", ""]

print("Normalizing HMDA missing indicators → null ...")
for col_name in df_eng.columns:
    if col_name == "label":
        continue  

    df_eng = df_eng.withColumn(
        col_name,
        F.when(
            F.col(col_name).cast("string").isin(MISSING_VALUES),
            F.lit(None)
        ).otherwise(F.col(col_name))
    )

print("✓ Converted Exempt/NA/1111/'' → null for all non-target columns")

# ── Show top missing columns ──
null_counts = df_eng.select([
    F.count(F.when(F.col(c).isNull(), True)).alias(c)
    for c in df_eng.columns if c != "label"
])
null_dict = null_counts.collect()[0].asDict()

print(f"\n{'Column':<45} {'Nulls':>12} {'% Missing':>10}")
print("-" * 70)

Normalizing HMDA missing indicators → null ...
✓ Converted Exempt/NA/1111/'' → null for all non-target columns



Column                                               Nulls  % Missing
----------------------------------------------------------------------


In [7]:
# ============================================================
# Create Missingness Indicator Features
# ============================================================

INFORMATIVE_MISSING_COLS = [
    "interest_rate",                    # Often missing for denied (no rate offered)
    "combined_loan_to_value_ratio",     # Missing when property value unknown
    "property_value",                   # Missing more for denied
    "income",                           # Missing more for denied
    "debt_to_income_ratio",             # String field, often "Exempt"
    "loan_term",                        # Missing for some denied apps
    "intro_rate_period",                # Mostly missing — but informative when present
    "applicant_credit_score_type",      # Credit score type varies by denial outcome
    "co_applicant_credit_score_type",   # Co-applicant presence is itself a signal
]

indicator_cols_created = []
for col_name in INFORMATIVE_MISSING_COLS:
    if col_name in df_eng.columns:
        ind_name = f"{col_name}_is_missing"
        df_eng = df_eng.withColumn(
            ind_name,
            F.when(F.col(col_name).isNull(), 1.0).otherwise(0.0)
        )
        indicator_cols_created.append(ind_name)

print(f"✓ Created {len(indicator_cols_created)} missingness indicator features:")
for c in indicator_cols_created:
    ct = df_eng.filter(F.col(c) == 1).count()
    print(f"  {c:<50} {ct:>10,} ({ct/target_total*100:.1f}%)")

✓ Created 9 missingness indicator features:
  interest_rate_is_missing                            2,211,322 (28.8%)
  combined_loan_to_value_ratio_is_missing               751,069 (9.8%)
  property_value_is_missing                             436,910 (5.7%)
  income_is_missing                                     345,895 (4.5%)
  debt_to_income_ratio_is_missing                       627,872 (8.2%)
  loan_term_is_missing                                  381,964 (5.0%)
  intro_rate_period_is_missing                        5,754,560 (74.9%)
  applicant_credit_score_type_is_missing                232,009 (3.0%)
  co_applicant_credit_score_type_is_missing             232,009 (3.0%)


## Section 5 — Numeric Feature Transformations

In [8]:
# ============================================================
# Cast Numeric Columns to DoubleType
# ============================================================

# Which columns should be numeric?
NUMERIC_FEATURES = [c for c in NUMERIC_CONTINUOUS
                    if c in df_eng.columns and c not in LEAKAGE_DROP]

# Plus our engineered numeric features
ADDITIONAL_NUMERIC = ["high_cost_flag"]
ALL_NUMERIC = NUMERIC_FEATURES + ADDITIONAL_NUMERIC

print("Casting columns to DoubleType ...")
cast_count = 0
for col_name in ALL_NUMERIC:
    if col_name not in df_eng.columns:
        continue
    current = str(df_eng.schema[col_name].dataType)
    if current not in ("DoubleType",):
        df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("double"))
        cast_count += 1

# Also ensure indicator columns are Double (they should already be)
for col_name in indicator_cols_created:
    df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("double"))

print(f"✓ Cast {cast_count} columns to DoubleType")
print(f"✓ Total numeric features: {len(ALL_NUMERIC)}")

Casting columns to DoubleType ...
✓ Cast 16 columns to DoubleType
✓ Total numeric features: 16


In [9]:
# ============================================================
#  Log Transform Skewed Numeric Features
# ============================================================


LOG_TRANSFORM_COLS = [
    "loan_amount",      # Right-skewed financial amount
    "income",           # Right-skewed income distribution
    "property_value",   # Right-skewed property values
    "tract_population", # Population varies hugely (rural vs urban)
]

print("=" * 80)
print("LOG TRANSFORMATIONS: log1p(x)")
print("=" * 80)
print(f"  {'Column':<30} {'Mean':>12} → {'Log Mean':>12}  {'Max':>14} → {'Log Max':>12}")
print("-" * 90)

log_cols_created = []
for col_name in LOG_TRANSFORM_COLS:
    if col_name not in df_eng.columns:
        continue

    new_col = f"{col_name}_log"
    df_eng = df_eng.withColumn(
        new_col,
        F.when(
            F.col(col_name).isNotNull() & (F.col(col_name) >= 0),
            F.log1p(F.col(col_name))
        ).otherwise(F.lit(None))
    )
    log_cols_created.append(new_col)

    # Report transformation effect
    s = df_eng.select(
        F.mean(col_name).alias("om"), F.max(col_name).alias("ox"),
        F.mean(new_col).alias("lm"), F.max(new_col).alias("lx")
    ).first()

    print(f"  {col_name:<30} {s['om'] or 0:>12.1f} → {s['lm'] or 0:>12.3f}  {s['ox'] or 0:>14.1f} → {s['lx'] or 0:>12.3f}")

print(f"\n✓ Created {len(log_cols_created)} log-transformed columns")
print("  Original columns kept for interpretability. Pipeline uses _log versions.")

LOG TRANSFORMATIONS: log1p(x)
  Column                                 Mean →     Log Mean             Max →      Log Max
------------------------------------------------------------------------------------------


  loan_amount                        313375.0 →       12.026  232323235000.0 →       26.171
  income                                183.8 →        4.617     132000000.0 →       18.698
  property_value                     534768.9 →       12.818    2147483647.0 →       21.488


  tract_population                     4760.3 →        8.284         30199.0 →       10.316

✓ Created 4 log-transformed columns
  Original columns kept for interpretability. Pipeline uses _log versions.


In [10]:
# ============================================================
# Clip Outliers & Create Engineered Features
# ============================================================

print("=" * 80)
print("OUTLIER CLIPPING & FEATURE ENGINEERING")
print("=" * 80)

# combined_loan_to_value_ratio: Valid range [0, 200%]
# Values > 200% are either data errors or "Exempt" codes surviving as numbers
if "combined_loan_to_value_ratio" in df_eng.columns:
    ct = df_eng.filter(F.col("combined_loan_to_value_ratio") > 200).count()
    df_eng = df_eng.withColumn(
        "combined_loan_to_value_ratio",
        F.when(F.col("combined_loan_to_value_ratio") > 200, 200.0)
        .when(F.col("combined_loan_to_value_ratio") < 0, F.lit(None))
        .otherwise(F.col("combined_loan_to_value_ratio"))
    )
    print(f"  CLTV: Clipped {ct:,} rows > 200% to 200")

# interest_rate: Valid range [0, 30%] — no real mortgage exceeds this
if "interest_rate" in df_eng.columns:
    ct = df_eng.filter(F.col("interest_rate") > 30).count()
    df_eng = df_eng.withColumn(
        "interest_rate",
        F.when(F.col("interest_rate") > 30, F.lit(None))
        .when(F.col("interest_rate") < 0, F.lit(None))
        .otherwise(F.col("interest_rate"))
    )
    print(f"  Interest rate: Nulled {ct:,} rows > 30% (likely errors)")

# ──── B. ENGINEERED FEATURES ────

if "loan_amount" in df_eng.columns and "income" in df_eng.columns:
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio",
        F.when(
            F.col("income").isNotNull() & (F.col("income") > 0),
            F.col("loan_amount") / F.col("income")
        ).otherwise(F.lit(None))
    )
    # Clip extreme ratios (>50 is unrealistic for any mortgage product)
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio",
        F.when(F.col("loan_to_income_ratio") > 50, 50.0)
        .otherwise(F.col("loan_to_income_ratio"))
    )
    # Also log-transform this ratio (it's skewed too)
    df_eng = df_eng.withColumn(
        "loan_to_income_ratio_log",
        F.when(F.col("loan_to_income_ratio").isNotNull() & (F.col("loan_to_income_ratio") > 0),
               F.log1p(F.col("loan_to_income_ratio")))
        .otherwise(F.lit(None))
    )
    print("  ✓ Created loan_to_income_ratio + loan_to_income_ratio_log")

# B2: is_joint_application — fundamentally changes risk profile
if "co_applicant_age" in df_eng.columns:
    df_eng = df_eng.withColumn(
        "is_joint_application",
        F.when(
            F.col("co_applicant_age").isNotNull() &
            ~F.col("co_applicant_age").cast("string").isin("9999", "8888"),
            1.0
        ).otherwise(0.0)
    )
    print("  ✓ Created is_joint_application (from co_applicant_age)")

# B3: Cast remaining numeric-looking columns
for c in ["tract_to_msa_income_percentage", "tract_minority_population_percent",
           "ffiec_msa_md_median_family_income", "tract_owner_occupied_units",
           "tract_one_to_four_family_homes", "tract_median_age_of_housing_units"]:
    if c in df_eng.columns:
        df_eng = df_eng.withColumn(c, F.col(c).cast("double"))

print(f"\n✓ DataFrame now has {len(df_eng.columns)} columns (including engineered features)")

OUTLIER CLIPPING & FEATURE ENGINEERING
  CLTV: Clipped 4,890 rows > 200% to 200
  Interest rate: Nulled 1 rows > 30% (likely errors)
  ✓ Created loan_to_income_ratio + loan_to_income_ratio_log
  ✓ Created is_joint_application (from co_applicant_age)

✓ DataFrame now has 59 columns (including engineered features)


## Section 6 — Categorical Encoding Strategy

In [11]:
# ============================================================
# Classify Categoricals by Cardinality
# ============================================================

ENGINEERED = (log_cols_created + indicator_cols_created +
              ["loan_to_income_ratio", "loan_to_income_ratio_log",
               "is_joint_application", "high_cost_flag"])

remaining_cats = [c for c in df_eng.columns
                  if c not in ALL_NUMERIC
                  and c not in ENGINEERED
                  and c != "label"]

print("=" * 80)
print("CATEGORICAL ENCODING PLAN")
print("=" * 80)
print(f"  {'Column':<43} {'Unique':>8} {'Strategy':<25}")
print("-" * 80)

CAT_ONEHOT = []       # 2-10 values → StringIndexer + OneHotEncoder
CAT_INDEX_ONLY = []   # 11-50 values → StringIndexer only
CAT_DROP_HIGH = []    # 50+ values → DROP (too many)

for col_name in sorted(remaining_cats):
    n_unique = df_eng.select(col_name).distinct().count()

    if n_unique <= 1:
        strategy = "DROP (zero variance)"
        CAT_DROP_HIGH.append(col_name)
    elif n_unique <= 10:
        strategy = "OneHot"
        CAT_ONEHOT.append(col_name)
    elif n_unique <= 50:
        strategy = "IndexOnly"
        CAT_INDEX_ONLY.append(col_name)
    else:
        strategy = "DROP (>50 unique)"
        CAT_DROP_HIGH.append(col_name)

    print(f"  {col_name:<43} {n_unique:>8} {strategy}")

print(f"\nSummary:")
print(f"  OneHot encoding:     {len(CAT_ONEHOT)} columns")
print(f"  Index-only encoding: {len(CAT_INDEX_ONLY)} columns")
print(f"  Drop (high card.):   {len(CAT_DROP_HIGH)} → {CAT_DROP_HIGH}")

if CAT_DROP_HIGH:
    df_eng = df_eng.drop(*CAT_DROP_HIGH)
    print(f"\n✓ Dropped {len(CAT_DROP_HIGH)} high-cardinality columns")

CATEGORICAL ENCODING PLAN
  Column                                        Unique Strategy                 
--------------------------------------------------------------------------------
  applicant_age                                      8 OneHot
  applicant_age_above_62                             3 OneHot
  applicant_credit_score_type                       11 IndexOnly
  aus_1                                              8 OneHot
  co_applicant_age                                   9 OneHot
  co_applicant_age_above_62                          3 OneHot
  co_applicant_credit_score_type                    12 IndexOnly
  conforming_loan_limit                              4 OneHot
  construction_method                                2 OneHot
  debt_to_income_ratio                              20 IndexOnly
  derived_dwelling_category                          4 OneHot
  derived_ethnicity                                  5 OneHot
  derived_loan_product_type                          8 OneH

In [12]:
# ============================================================
# Prepare Categoricals for StringIndexer
# ============================================================

ALL_CATEGORICALS = CAT_ONEHOT + CAT_INDEX_ONLY

cast_ct = 0
for col_name in ALL_CATEGORICALS:
    if col_name in df_eng.columns:
        if str(df_eng.schema[col_name].dataType) != "StringType":
            df_eng = df_eng.withColumn(col_name, F.col(col_name).cast("string"))
            cast_ct += 1

# Fill null categoricals with "Unknown"
for col_name in ALL_CATEGORICALS:
    if col_name in df_eng.columns:
        df_eng = df_eng.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), "Unknown").otherwise(F.col(col_name))
        )

print(f"✓ Cast {cast_ct} categorical columns to StringType")
print(f"✓ Filled null categoricals with 'Unknown'")
print(f"✓ Total categoricals for encoding: {len(ALL_CATEGORICALS)}")

✓ Cast 25 categorical columns to StringType
✓ Filled null categoricals with 'Unknown'
✓ Total categoricals for encoding: 25


## Section 7 — PySpark ML Pipeline Assemble

In [13]:
# ============================================================
# Build Pipeline Stages
# ============================================================

start = time.time()

from pyspark.sql.types import DoubleType, FloatType

numeric_to_impute = []
for col_name in df_eng.columns:
    if col_name == "label" or col_name in ALL_CATEGORICALS:
        continue
    col_type = df_eng.schema[col_name].dataType
    if isinstance(col_type, (DoubleType, FloatType)):
        numeric_to_impute.append(col_name)

print(f"Found {len(numeric_to_impute)} numeric columns for imputation:")
for c in numeric_to_impute:
    print(f"  {c}")

# Safety check — Imputer crashes with empty inputCols
assert len(numeric_to_impute) > 0, \
    f"No numeric columns found! Check Cell 7 casting. " \
    f"Column dtypes: {[(c, str(df_eng.schema[c].dataType)) for c in df_eng.columns[:10]]}"

imputer_output = [f"{c}_imp" for c in numeric_to_impute]

imputer = Imputer(
    inputCols=numeric_to_impute,
    outputCols=imputer_output,
    strategy="median"  # Robust to skew — better than mean for financial data
)
print(f"\nStage 1: Imputer — {len(numeric_to_impute)} numeric columns → median")

# ──── STAGE 2: StringIndexer for ALL categoricals ────
indexers = []
indexed_col_names = []

for col_name in ALL_CATEGORICALS:
    if col_name not in df_eng.columns:
        continue
    out = f"{col_name}_idx"
    indexers.append(StringIndexer(
        inputCol=col_name, outputCol=out,
        handleInvalid="keep"  # Unseen categories → special index (won't crash)
    ))
    indexed_col_names.append(out)

print(f"Stage 2: StringIndexer × {len(indexers)}")

# ──── STAGE 3: OneHotEncoder for LOW cardinality only ────
ohe_in  = [f"{c}_idx" for c in CAT_ONEHOT if c in df_eng.columns]
ohe_out = [f"{c}_ohe" for c in CAT_ONEHOT if c in df_eng.columns]

encoder = OneHotEncoder(
    inputCols=ohe_in, outputCols=ohe_out,
    dropLast=True  # Avoids dummy variable trap
)
print(f"Stage 3: OneHotEncoder × {len(ohe_in)}")

# ──── STAGE 4: VectorAssembler ────
# Collect ALL feature columns for the final vector
feature_cols = []
feature_cols.extend(imputer_output)                                            # Imputed numerics
feature_cols.extend(ohe_out)                                                    # OneHot vectors
feature_cols.extend([f"{c}_idx" for c in CAT_INDEX_ONLY if c in df_eng.columns])  # Index-only cats

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_unscaled",
    handleInvalid="skip"  # Drop rows with ANY remaining null
)
print(f"Stage 4: VectorAssembler — {len(feature_cols)} inputs → 'features_unscaled'")

# ──── STAGE 5: StandardScaler ────
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=False,  # CRITICAL: sparse + withMean=True = dense = OOM
    withStd=True     # Normalize to unit variance
)
print(f"Stage 5: StandardScaler → 'features'")

# ──── ASSEMBLE PIPELINE ────
pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler])

print(f"\n{'='*60}")
print(f"PIPELINE ASSEMBLED: {len(pipeline.getStages())} total stages")
print(f"{'='*60}")
print(f"  Imputed numerics:       {len(imputer_output)}")
print(f"  OneHot encoded:         {len(ohe_out)}")
print(f"  Index-only cats:        {len([f'{c}_idx' for c in CAT_INDEX_ONLY if c in df_eng.columns])}")
print(f"  Total assembler inputs: {len(feature_cols)}")
print(f"  Construction time:      {time.time()-start:.1f}s")

Found 32 numeric columns for imputation:
  loan_amount
  combined_loan_to_value_ratio
  interest_rate
  loan_term
  intro_rate_period
  property_value
  multifamily_affordable_units
  income
  tract_population
  tract_minority_population_percent
  ffiec_msa_md_median_family_income
  tract_to_msa_income_percentage
  tract_owner_occupied_units
  tract_one_to_four_family_homes
  tract_median_age_of_housing_units
  high_cost_flag
  interest_rate_is_missing
  combined_loan_to_value_ratio_is_missing
  property_value_is_missing
  income_is_missing
  debt_to_income_ratio_is_missing
  loan_term_is_missing
  intro_rate_period_is_missing
  applicant_credit_score_type_is_missing
  co_applicant_credit_score_type_is_missing
  loan_amount_log
  income_log
  property_value_log
  tract_population_log
  loan_to_income_ratio
  loan_to_income_ratio_log
  is_joint_application

Stage 1: Imputer — 32 numeric columns → median
Stage 2: StringIndexer × 25
Stage 3: OneHotEncoder × 22
Stage 4: VectorAssembler — 5

## Section 8 — Stratified Train/Test Split & Pipeline Fitting

In [14]:
# ============================================================
# Stratified Train/Test Split
# ============================================================

start = time.time()

TRAIN_FRACTION = 0.8
SEED = 42  # Fixed for reproducibility

train_df = df_eng.sampleBy("label", fractions={0.0: TRAIN_FRACTION, 1.0: TRAIN_FRACTION}, seed=SEED)
test_df  = df_eng.subtract(train_df)  # Complement

# Verify
train_total   = train_df.count()
train_denied  = train_df.filter(F.col("label") == 1).count()
train_orig    = train_total - train_denied

test_total    = test_df.count()
test_denied   = test_df.filter(F.col("label") == 1).count()
test_orig     = test_total - test_denied

print("=" * 70)
print("STRATIFIED TRAIN/TEST SPLIT")
print("=" * 70)
print(f"  {'Set':<16} {'Total':>12} {'Originated':>12} {'Denied':>12} {'Denial %':>10}")
print("-" * 65)
print(f"  {'Train (80%)':<16} {train_total:>12,} {train_orig:>12,} {train_denied:>12,} {train_denied/train_total*100:>9.1f}%")
print(f"  {'Test  (20%)':<16} {test_total:>12,} {test_orig:>12,} {test_denied:>12,} {test_denied/test_total*100:>9.1f}%")
print(f"  {'Full':<16} {target_total:>12,} {originated_count:>12,} {denied_count:>12,} {denied_count/target_total*100:>9.1f}%")
print(f"\n✓ Split in {time.time()-start:.1f}s")
print(f"  Denial rate preserved: train={train_denied/train_total*100:.2f}% vs test={test_denied/test_total*100:.2f}%")

STRATIFIED TRAIN/TEST SPLIT
  Set                     Total   Originated       Denied   Denial %
-----------------------------------------------------------------
  Train (80%)         6,148,985    4,553,462    1,595,523      25.9%
  Test  (20%)         1,533,558    1,136,116      397,442      25.9%
  Full                7,686,484    5,691,726    1,994,758      26.0%

✓ Split in 127.9s
  Denial rate preserved: train=25.95% vs test=25.92%


In [15]:
# ============================================================
# Fit Pipeline on TRAINING Data Only — Then Transform Both
# ============================================================

start = time.time()

print("=" * 70)
print("FITTING PIPELINE ON TRAINING DATA")
print("=" * 70)
print(f"Training records:  {train_total:,}")
print(f"Pipeline stages:   {len(pipeline.getStages())}")
print("\nFitting... (may take 2-5 minutes depending on data size)")

pipeline_model = pipeline.fit(train_df)
fit_time = time.time() - start
print(f"\n✓ Pipeline fitted in {fit_time:.1f}s")

# ── Transform both sets ──
print("\nTransforming train set...")
t0 = time.time()
train_transformed = pipeline_model.transform(train_df)
print(f"  ✓ Done in {time.time()-t0:.1f}s")

print("Transforming test set...")
t0 = time.time()
test_transformed = pipeline_model.transform(test_df)
print(f"  ✓ Done in {time.time()-t0:.1f}s")

# Keep only what the model needs: label + features
train_final = train_transformed.select("label", "features")
test_final  = test_transformed.select("label", "features")

# Cache & materialize
train_final.cache()
test_final.cache()
train_count = train_final.count()
test_count  = test_final.count()

train_dim = train_final.first()['features'].size
print(f"\n✓ Final datasets ready:")
print(f"  Train: {train_count:,} rows")
print(f"  Test:  {test_count:,} rows")
print(f"  Feature vector size: {train_dim}")

FITTING PIPELINE ON TRAINING DATA
Training records:  6,148,985
Pipeline stages:   29

Fitting... (may take 2-5 minutes depending on data size)



✓ Pipeline fitted in 56.1s

Transforming train set...
  ✓ Done in 0.8s
Transforming test set...
  ✓ Done in 1.3s


26/03/02 00:39:19 WARN MemoryStore: Not enough space to cache rdd_769_6 in memory! (computed 260.0 MiB so far)
26/03/02 00:39:19 WARN MemoryStore: Not enough space to cache rdd_769_1 in memory! (computed 260.0 MiB so far)
26/03/02 00:39:19 WARN BlockManager: Persisting block rdd_769_1 to disk instead.
26/03/02 00:39:19 WARN BlockManager: Persisting block rdd_769_6 to disk instead.
26/03/02 00:40:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/02 00:40:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/02 00:40:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/02 00:40:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/02 00:40:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/02 00:40:11 WARN RowBased


✓ Final datasets ready:
  Train: 6,148,985 rows
  Test:  1,533,558 rows
  Feature vector size: 145


## Section 9 — Post-Pipeline Verification & Quality Checks

In [16]:
# ============================================================
# Post-Pipeline Quality Checks
# ============================================================

print("=" * 70)
print("POST-PIPELINE VERIFICATION")
print("=" * 70)

# ── CHECK 1: Row survival ──
processed = train_count + test_count
dropped_pct = (1 - processed / target_total) * 100
print(f"\n1. ROW COUNT")
print(f"   Original (filtered):  {target_total:,}")
print(f"   After pipeline:       {processed:,}")
print(f"   Rows dropped:         {target_total - processed:,} ({dropped_pct:.1f}%)")
if dropped_pct < 5:
    print("   ✓ Acceptable (< 5%)")
else:
    print("   ⚠ WARNING: >5% rows lost — investigate VectorAssembler nulls")

# ── CHECK 2: Feature dimensions ──
test_dim = test_final.first()['features'].size
print(f"\n2. FEATURE DIMENSIONS")
print(f"   Train: {train_dim}  |  Test: {test_dim}")
if train_dim == test_dim:
    print("   ✓ Match")
else:
    print("   ✗ MISMATCH — Pipeline bug!")

# ── CHECK 3: Class balance preservation ──
train_denial_rate = train_final.filter(F.col("label") == 1).count() / train_count
test_denial_rate  = test_final.filter(F.col("label") == 1).count() / test_count
print(f"\n3. CLASS BALANCE")
print(f"   Train denial rate: {train_denial_rate*100:.2f}%")
print(f"   Test denial rate:  {test_denial_rate*100:.2f}%")
print(f"   Difference:        {abs(train_denial_rate - test_denial_rate)*100:.2f}%")
if abs(train_denial_rate - test_denial_rate) < 0.5:
    print("   ✓ Preserved")
else:
    print("   ⚠ Drift detected")

# ── CHECK 4: Sample vectors ──
print(f"\n4. SAMPLE FEATURE VECTORS (first 5 rows, first 10 values)")
for row in train_final.take(5):
    vals = row['features'].toArray()[:10]
    vals_str = ", ".join([f"{v:.3f}" for v in vals])
    print(f"   label={int(row['label'])} | [{vals_str}, ...]")

# ── CHECK 5: Feature statistics ──
print(f"\n5. FEATURE STATISTICS")
summary = train_final.select(
    Summarizer.metrics("mean", "variance").summary(F.col("features"))
).collect()[0][0]

means = summary[0].toArray()
variances = summary[1].toArray()
zero_var = sum(1 for v in variances if v == 0)
print(f"   Total features:           {len(means)}")
print(f"   Zero-variance features:   {zero_var}")
print(f"   Mean range:  [{min(means):.4f}, {max(means):.4f}]")
print(f"   Var range:   [{min(variances):.4f}, {max(variances):.4f}]")
if zero_var == 0:
    print("   ✓ All features have non-zero variance")
else:
    print(f"   ⚠ {zero_var} zero-variance features found — consider removing")

POST-PIPELINE VERIFICATION

1. ROW COUNT
   Original (filtered):  7,686,484
   After pipeline:       7,682,543
   Rows dropped:         3,941 (0.1%)
   ✓ Acceptable (< 5%)

2. FEATURE DIMENSIONS
   Train: 145  |  Test: 145
   ✓ Match



3. CLASS BALANCE
   Train denial rate: 25.95%
   Test denial rate:  25.92%
   Difference:        0.03%
   ✓ Preserved

4. SAMPLE FEATURE VECTORS (first 5 rows, first 10 values)
   label=1 | [0.000, 2.692, 4.806, 2.119, 0.042, 0.065, 0.000, 0.002, 2.267, 3.145, ...]
   label=1 | [0.001, 4.386, 4.806, 4.237, 0.042, 0.033, 0.000, 0.001, 1.393, 3.530, ...]
   label=0 | [0.003, 4.521, 4.632, 4.237, 0.042, 0.062, 0.000, 0.001, 2.308, 3.222, ...]
   label=0 | [0.001, 2.790, 5.156, 4.237, 0.042, 0.059, 0.000, 0.001, 1.945, 3.297, ...]
   label=1 | [0.001, 4.295, 4.806, 4.237, 0.042, 0.028, 0.000, 0.000, 1.762, 3.807, ...]

5. FEATURE STATISTICS
   Total features:           145
   Zero-variance features:   0
   Mean range:  [0.0021, 149.1943]
   Var range:   [1.0000, 1.0000]
   ✓ All features have non-zero variance


In [17]:
# ============================================================
# Save Processed Data & Pipeline Model
# ============================================================

import os

# Save train/test Parquet
train_path = os.path.join(OUTPUT_DIR, "train_features.parquet")
test_path  = os.path.join(OUTPUT_DIR, "test_features.parquet")

train_final.write.mode("overwrite").parquet(train_path)
# print(f"✓ Train saved: {train_path}")

test_final.write.mode("overwrite").parquet(test_path)
# print(f"✓ Test saved:  {test_path}")

# Save pipeline model for production use
model_path = os.path.join(OUTPUT_DIR, "pipeline_model")
pipeline_model.write().overwrite().save(model_path)
# print(f"✓ Pipeline model saved: {model_path}")

# Save feature metadata
feature_meta = {
    "n_features": train_dim,
    "n_train": train_count,
    "n_test": test_count,
    "train_denial_rate": float(train_denial_rate),
    "test_denial_rate": float(test_denial_rate),
    "numeric_features": numeric_to_impute,
    "onehot_categoricals": CAT_ONEHOT,
    "index_categoricals": CAT_INDEX_ONLY,
    "indicator_features": indicator_cols_created,
    "log_features": log_cols_created,
    "assembler_inputs": feature_cols,
}
meta_path = os.path.join(OUTPUT_DIR, "feature_metadata.json")
with open(meta_path, "w") as f:
    json.dump(feature_meta, f, indent=2)
# print(f"✓ Feature metadata saved: {meta_path}")

26/03/02 00:40:55 WARN DAGScheduler: Broadcasting large task binary with size 1017.2 KiB
